# Enveda CASMI 2026 — Simple MS Simulator baseline (ICEBERG virtual library, on the fly)

**Generated by `build/mk_simple_sim.py` — do not hand-edit.** Spec: `specs_simple_ms_simulator/`
(mission, roadmap Phase 4/5, plan `here-i-am-working-pure-kernighan.md`).

Settings: **GPU T4**, **Internet OFF**. Inputs: the competition data, **casmi26-iceberg-offline**
(wheelhouse + MassSpecGym `msg_all` checkpoints), the COCONUT dataset phase1 uses (`coco_meta.pkl`,
`coco_mass.npy`, `coco_fp.npy`), the dataset with `fp_bits.npy`, and the RDKit 2026.03.3 wheel.

Per molecule: phase1's centred ±8.5 ppm window (+ PPM_EXPAND tiers, mass-nearest backfill), then a
**two-tier ranking** — tier 1 = real-library entropy match `lv ≥ τ` (by `lv`); tier 2 = the rest, by
`sv` = entropy similarity of the query to the candidate's **ICEBERG-simulated spectrum at the query's
adduct and exact CE** (merged CE lists: max over the constituents), then MetFrag-lite `fr`, then |ppm|.
Adducts ICEBERG does not cover (G2b) use `fr` only. No FPNet, no analog channel, no learned ranker.

Phase 1 fixed the design: tolerance 0 eV (a grid is useless), 36.6 mol/s on one T4 (a 2nd GPU adds
nothing), so everything is simulated here (option A). A watchdog simulates candidates in priority order
and stops before the budget, so an oversized hidden test degrades instead of timing out.

In [ ]:
class CFG:
    # --- phase1 values (the embedded phase1 cells read these) -------------------------------------
    PPM_WIN = 8.5
    PPM_CENTER = 1.45
    PPM_EXPAND = (8.5, 20.0, 45.0, 80.0)
    PPM_FALLBACK = 30.0
    INT_FLOOR = 0.002
    MAX_PEAKS = 64
    MZ_TOL = 0.01
    INT_POWER = 1.0
    ENT_WEIGHT = True
    ANALOG_WIN = 200.0          # analog_sim rides along in the evidence cell; unused here
    N_ANALOG = 100
    POLARITY_ANALOG = True
    USE_BIO_DB = False
    TOPN = 25
    STRUCT_MASS = True
    MAX_BONDS = 60
    DEDUP_KEYS = True
    EXPAND_BACKFILL = True
    BACKFILL_PPM = 120.0
    ALLOW_FAST = True
    FAST_DEV_MOLECULES = 30
    PUBLIC_N_MOL = 400
    PUBLIC_N_SPEC = 1213

    # --- simulator baseline ---------------------------------------------------------------------------
    TAU = 0.75                  # tier-1 threshold on real-library entropy similarity (untuned by design)
    MERGE_RULE = "max"          # merged CE lists: "max" | "sum" | "mid" (user decision 2026-09-27: max)
    COVERED = ("[M+H]+", "[M+NH4]+", "[M+Na]+", "[M-H]-", "[M+CH2O2-H]-", "[M+Cl]-")   # G2b, gate_report
    CE_MODE = "ev"              # Phase 0: msg_all / eV / QTOF
    INSTRUMENT = "QTOF"
    CE_DEFAULT = 40.0           # a covered spectrum with no usable CE (counted in the report)
    BATCH = 128                 # t4.json: bs 128 best on one T4
    BATCH_RETRY = 32
    CHUNK = 20_000              # worker rows per call
    SPARSE_K = 100
    T4_RATE = 36.6              # t4.json, mol/s (replaced by the measured rate after the first chunk)
    STARTUP_S = 10.4            # t4.json, worker start-up per call
    HARD_BUDGET_H = 7.5         # simulation stops so that pass 2 + submission finish before this
    PASS2_S_PER_MOL = 1.0       # reserve for pass 2 (+ 300 s)
    SIM_TIER1 = False           # simulate tier-1 candidates too (forced True in a fast commit)
    MIN_OK_FAST = 0.99          # fast commit: required share of requests predicted
    FORCE_FULL = False          # debugging only: full run even on the public test file
    USE_SIM = True              # False = the no-simulation control (sv = 0 everywhere; same code path otherwise)

In [ ]:
import os, sys, glob, time, json, shutil, pickle, math, subprocess
from multiprocessing import Pool as MPool
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

T0 = time.time()
TIMES = {}

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f"File not found in /kaggle/input: {name}")
    return sorted(hits, key=len)[0]

def find_dir(name):
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(h)]
    if not hits:
        raise FileNotFoundError(f"Directory not found in /kaggle/input: {name} -- is the dataset attached?")
    return sorted(hits, key=len)[0]

COMP   = os.path.dirname(find_file('test.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
TEST   = os.path.join(COMP, 'test.parquet')
SAMPLE = os.path.join(COMP, 'sample_submission.csv')
print(f"[INFO] Competition directory: {COMP}")

rdkit_whls = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if rdkit_whls:
    print(f"[INFO] Installing offline RDKit wheel: {rdkit_whls[0]}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', rdkit_whls[0]], check=False)
try:
    import rdkit
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
    from rdkit.Chem.Descriptors import ExactMolWt
    RDLogger.DisableLog('rdApp.*')
    HAVE_RDKIT = True
except Exception as e:
    HAVE_RDKIT = False
    print(f"[ERROR] RDKit unavailable ({e})")
# without RDKit the pool loses every train structure and MetFrag-lite is off -- never ship that
assert HAVE_RDKIT, "RDKit is required: attach the casmi26-offline-rdkit-2026033 wheel dataset"
print(f"[INFO] RDKit {rdkit.__version__}")

## phase1 code, embedded verbatim (numba kernels · adduct physics + library · candidate pool · evidence)

In [ ]:
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n = len(mz)
    if n == 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mx = 0.0
    for i in range(n):
        if it[i] > mx: mx = it[i]
    if mx <= 0:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    thr = floor * mx
    c = 0
    for i in range(n):
        if it[i] >= thr: c += 1
    idx = np.empty(c, np.int64)
    j = 0
    for i in range(n):
        if it[i] >= thr:
            idx[j] = i; j += 1
    if c > topk:
        v = np.empty(c, np.float32)
        for i in range(c): v[i] = it[idx[i]]
        o = np.argsort(v)[c - topk:]
        k2 = np.empty(topk, np.int64)
        for i in range(topk): k2[i] = idx[o[i]]
        k2.sort()
        idx = k2
        c = topk
    om = np.empty(c, np.float32)
    oi = np.empty(c, np.float32)
    s = 0.0
    for i in range(c):
        om[i] = mz[idx[i]]
        v = it[idx[i]] ** power
        oi[i] = v
        s += v
    if s > 0:
        for i in range(c): oi[i] /= s
    if ent_weight:
        S = 0.0
        for i in range(c):
            if oi[i] > 0: S -= oi[i] * np.log(oi[i])
        if S < 3.0:
            w = 0.25 + 0.25 * S
            s2 = 0.0
            for i in range(c):
                oi[i] = oi[i] ** w
                s2 += oi[i]
            if s2 > 0:
                for i in range(c): oi[i] /= s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz, qp, cmz, cp, tol):
    i = 0; j = 0; n = len(qmz); m = len(cmz)
    SA = 0.0
    for x in range(n):
        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])
    SB = 0.0
    for x in range(m):
        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])
    SAB = 0.0; tot = 0.0
    buf = np.empty(n + m, np.float64); b = 0
    while i < n and j < m:
        d = qmz[i] - cmz[j]
        if d < -tol:
            buf[b] = qp[i]; i += 1; b += 1
        elif d > tol:
            buf[b] = cp[j]; j += 1; b += 1
        else:
            buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1
    while i < n: buf[b] = qp[i]; i += 1; b += 1
    while j < m: buf[b] = cp[j]; j += 1; b += 1
    for x in range(b): tot += buf[x]
    if tot <= 0: return 0.0
    for x in range(b):
        v = buf[x] / tot
        if v > 0: SAB -= v * np.log(v)
    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):
    """Evaluates max of direct and mass-shifted spectral entropy similarity."""
    a = entropy_sim(qmz, qp, cmz, cp, tol)
    if shift > -0.001 and shift < 0.001:
        return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)):
        sm[i] = cmz[i] + shift
    b = entropy_sim(qmz, qp, sm, cp, tol)
    return a if a > b else b

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim(qmz, qp, cm, cp, tol)
    return out

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):
    out = np.zeros(len(cand), np.float32)
    for k in prange(len(cand)):
        c = cand[k]; a = off[c]; b = off[c + 1]
        if b <= a: continue
        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)
        if len(cm) == 0: continue
        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])
    return out

In [ ]:
# ===================================================================================
#  High-Precision Adduct Physics & Reference Library Construction
# ===================================================================================
MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,
            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,
            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)
E = 0.00054857990
PROTON = MASS['H'] - E
H2O = 2 * MASS['H'] + MASS['O']
NH4 = MASS['N'] + 4 * MASS['H']
FORMATE = MASS['C'] + 2 * MASS['H'] + 2 * MASS['O']
ACETATE = 2 * MASS['C'] + 4 * MASS['H'] + 2 * MASS['O']

ADDUCTS = {
    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS['Na'] - E),
    "[M+K]+": (1, 1, MASS['K'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2*H2O),
    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),
    "[M+CH3OH+H]+": (1, 1, PROTON + MASS['C'] + 4 * MASS['H'] + MASS['O']),
    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS['C'] + 3 * MASS['H'] + MASS['N']),
    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),
    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS['Cl'] + E), "[M]-": (1, 1, E),
    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS['Na'] - 2 * PROTON),
    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS['Na'] - E), "[2M+NH4]+": (2, 1, NH4 - E),
    "[2M+K]+": (2, 1, MASS['K'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),
    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS['Na'] - 2 * PROTON),
    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),
}

# --- TIER-1: exact neutral mass from the labelled molecular formula -----------------
# Far faster than 275k RDKit parses and exactly what the label implies.
import re as _re
_MONO = {"C":12.0,"H":1.0078250319,"N":14.0030740052,"O":15.9949146221,"S":31.97207069,
         "P":30.97376151,"F":18.99840320,"Cl":34.96885271,"Br":78.9183376,"I":126.904468,
         "Si":27.9769265327,"Se":79.9165196,"B":11.0093055,"Na":22.98976928,"K":38.96370649,
         "Fe":55.9349375,"D":2.0141017778,"As":74.9215965,"Li":7.0160034,"Sn":119.9021947}
_TOK = _re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    if not isinstance(f, str): return np.nan
    s, pos = 0.0, 0
    for m in _TOK.finditer(f):
        if m.start() != pos: return np.nan
        pos = m.end()
        el = m.group(1)
        if el not in _MONO: return np.nan
        s += _MONO[el] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan


def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan)
    ad = np.asarray(adduct, dtype=object)
    for a, (n, z, d) in ADDUCTS.items():
        m = (ad == a)
        if m.any():
            out[m] = (mz[m] * z - d) / n
    return out

def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14', 'normalized_smiles', 'adduct', 'ionization_mode',
                                     'precursor_mz', 'molecular_formula',
                                     'ms2_mzs', 'ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks()
    itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    
    # Phase 2: Polarity extraction (True = positive, False = negative)
    imode = np.asarray(t.column('ionization_mode').cast(pa.string()).to_pylist(), dtype=object)
    is_pos = np.array([str(m).lower() == 'positive' for m in imode], dtype=bool)

    nm_meas = neutral_mass(prec, add)
    if CFG.STRUCT_MASS:
        # TIER-1: prefer the label's exact mass; fall back to the measured one where the
        # formula is unparseable (0.46% of rows) or contains an exotic element.
        formula   = np.asarray(t.column('molecular_formula').cast(pa.string()).to_pylist(), dtype=object)
        _fm       = {f: formula_mass(f) for f in set(formula.tolist())}
        nm_struct = np.array([_fm.get(f, np.nan) for f in formula], dtype=np.float64)
        nm        = np.where(np.isfinite(nm_struct), nm_struct, nm_meas)
        n_fixed   = int(np.isfinite(nm_struct).sum())
        print(f"[TIER-1] structure-derived neutral mass for {n_fixed:,}/{len(nm):,} spectra "
              f"({n_fixed/len(nm)*100:.1f}%); {len(nm)-n_fixed:,} fell back to precursor-derived.",
              flush=True)
    else:
        nm = nm_meas
    ok  = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best:
            best[k] = s
    print(f"[INFO] Reference Library loaded: {len(off)-1:,} spectra / {len(best):,} unique structures ({time.time()-t0:.1f}s)", flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, is_pos=is_pos, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target - tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target + tol, 'right')
    return L['order'][lo:hi]

def build_rep(L):
    """Phase 2: Extracts richest representative spectrum per unique (scaffold, polarity) pair."""
    npk = np.diff(L['off'])
    best = {}
    ik = L['ik']
    is_pos = L.get('is_pos', np.ones(len(ik), dtype=bool))
    if not getattr(CFG, 'POLARITY_ANALOG', True):
        is_pos = np.ones(len(ik), dtype=bool)   # one rep per scaffold (tier-1 behaviour)
    
    for i in range(len(ik)):
        k = ik[i]
        if k:
            rep_key_tuple = (k, bool(is_pos[i]))
            if rep_key_tuple not in best or npk[i] > npk[best[rep_key_tuple]]:
                best[rep_key_tuple] = i

    rep_idx = np.array(sorted(best.values()))
    nm = L['nm'][rep_idx]
    ok = np.isfinite(nm)
    rep_idx = rep_idx[ok]
    nm = nm[ok]
    rep_key = ik[rep_idx]
    rep_is_pos = is_pos[rep_idx]
    o = np.argsort(nm)
    
    print(f"[INFO] Built polarity-aware representative analog set: {len(rep_idx):,} spectra across {len(set(rep_key)):,} unique scaffolds.", flush=True)
    return rep_idx[o], rep_key[o], nm[o], rep_is_pos[o]

In [ ]:
# ===================================================================================
#  711,705-Structure Candidate Pool with Bitpacked Precomputed Fingerprints
# ===================================================================================
BITS = np.load(find_file('fp_bits.npy'))
_g = {}

def _fp_init():
    if not HAVE_RDKIT: return
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not HAVE_RDKIT: return None
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([
            _g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
            np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)
        ])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class CandidatePool:
    """Unified candidate pool sorted by neutral exact mass."""
    def __init__(self, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        self._fp = fp[o]
        self.mass = mass[o]
        self.keys = np.asarray(keys, dtype=object)[o]
        self.smiles = np.asarray(smiles, dtype=object)[o]
        self.nbits = nbits
        self.k2i = {k: i for i, k in enumerate(self.keys)}

    def window(self, t, ppm):
        a = np.searchsorted(self.mass, t * (1 - ppm / 1e6), 'left')
        b = np.searchsorted(self.mass, t * (1 + ppm / 1e6), 'right')
        return np.arange(a, b)

    def fps(self, idx):
        return np.unpackbits(np.asarray(self._fp[idx]), axis=1)[:, :self.nbits]

def build_candidate_pool():
    t0 = time.time()
    d = os.path.dirname(find_file('coco_fp.npy'))
    cm = pickle.load(open(os.path.join(d, 'coco_meta.pkl'), 'rb'))
    co_fp = np.load(os.path.join(d, 'coco_fp.npy'))
    co_mass = np.load(os.path.join(d, 'coco_mass.npy'))
    raw_keys = np.asarray(cm['keys'], dtype=object)
    co_smis = np.asarray(cm['smiles'], dtype=object)

    # 1. Deduplicate within COCONUT by InChIKey14 (synchronized across all 4 arrays)
    co_k14_all = np.array([str(k)[:14] for k in raw_keys], dtype=object)
    _, first_idx = np.unique(co_k14_all, return_index=True)
    first_idx = np.sort(first_idx)

    co_k14  = co_k14_all[first_idx]
    co_smis = co_smis[first_idx]
    co_mass = co_mass[first_idx]
    co_fp   = co_fp[first_idx]
    print(f"[INFO] COCONUT 2.0 component: {len(raw_keys):,} -> {len(co_k14):,} unique InChIKey14 ({time.time()-t0:.1f}s)", flush=True)

    if CFG.USE_BIO_DB:
        try:
            bd = os.path.dirname(find_file('bio_fp.npy'))
            bm = pickle.load(open(os.path.join(bd, 'bio_meta.pkl'), 'rb'))
            bi_fp = np.load(os.path.join(bd, 'bio_fp.npy'))
            bi_mass = np.load(os.path.join(bd, 'bio_mass.npy'))
            bi_k14_all = np.array([str(k)[:14] for k in bm['keys']], dtype=object)
            bi_smis = np.asarray(bm['smiles'], dtype=object)

            _, bi_first_idx = np.unique(bi_k14_all, return_index=True)
            bi_first_idx = np.sort(bi_first_idx)
            bi_k14 = bi_k14_all[bi_first_idx]
            bi_smis = bi_smis[bi_first_idx]
            bi_mass = bi_mass[bi_first_idx]
            bi_fp = bi_fp[bi_first_idx]

            coco_set = set(co_k14)
            bi_mask = np.array([k not in coco_set for k in bi_k14], bool)
            co_fp = np.vstack([co_fp, bi_fp[bi_mask]])
            co_mass = np.concatenate([co_mass, bi_mass[bi_mask]])
            co_k14 = np.concatenate([co_k14, bi_k14[bi_mask]])
            co_smis = np.concatenate([co_smis, bi_smis[bi_mask]])
            print(f"[INFO] + ChEBI/LIPID MAPS: {bi_mask.sum():,} unique structures integrated.", flush=True)
        except Exception:
            print("[INFO] Bio DB not found, continuing with COCONUT + Train.", flush=True)

    # 2. Filter Train structures already in COCONUT (using matching 14-char keys)
    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    coco_k14_set = set(co_k14)
    tr = tr[~tr.inchikey14.isin(coco_k14_set)]
    print(f"[INFO] Computing fingerprints for {len(tr):,} novel training structures (multiprocessing)...", flush=True)

    if HAVE_RDKIT:
        with MPool(4) as mp:
            res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
        ok = [i for i, r in enumerate(res) if r is not None]
        tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
        tr_mass = np.array([res[i][1] for i in ok])
        tr_keys = tr.inchikey14.values[ok].astype(object)
        tr_smi = tr.normalized_smiles.values[ok].astype(object)
    else:
        tr_fp = np.empty((0, co_fp.shape[1]), dtype=np.uint8)
        tr_mass = np.empty(0, dtype=np.float64)
        tr_keys = np.empty(0, dtype=object)
        tr_smi = np.empty(0, dtype=object)

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_k14, tr_keys])
    smis = np.concatenate([co_smis, tr_smi])
    good = np.isfinite(mass)

    pool = CandidatePool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])
    print(f"[SUCCESS] Unified candidate pool ready: {len(pool.mass):,} structures (all unique InChIKey14) in {time.time()-t0:.1f}s", flush=True)
    return pool

In [ ]:
# ===================================================================================
#  Evidence Channels: Direct Library Match, Analog Match & MetFrag-lite
# ===================================================================================
AMU = {'C':12.0, 'H':1.00782503207, 'N':14.0030740048, 'O':15.9949146196, 'P':30.97376163,
       'S':31.97207100, 'F':18.99840322, 'Cl':34.96885268, 'Br':78.9183371, 'I':126.904473,
       'Na':22.9897692809, 'K':38.96370668, 'Si':27.9769265325, 'B':11.0093054, 'Se':79.9165213}
H_ATOM = AMU['H']
PROTON_MASS = H_ATOM - 0.00054857990

def clean_spectrum(mz, it):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)

def lib_sim(L, specs, target):
    """CLASS 1: Direct match against library spectra within tight neutral mass window."""
    cand = lib_window(L, target, target * CFG.PPM_WIN / 1e6)
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0):
                agg[k] = float(s)
    return agg

def analog_sim(L, specs, target, rep, rep_key, rep_nm, rep_pos=None):
    """CLASS 2: Mass-shifted spectral entropy match across a wide +-200 Da window.

    Polarity is handled upstream: build_rep keeps the richest representative per
    (scaffold, polarity) pair. Candidates are NEVER masked out by query polarity,
    because agg[k] already takes the max over that scaffold's representatives - the
    better-matching polarity wins on its own. Coverage is therefore always >= the
    one-representative-per-scaffold behaviour it replaces, never less.
    """
    lo = np.searchsorted(rep_nm, target - CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target + CFG.ANALOG_WIN, 'right')
    if hi <= lo: return []
    cand = rep[lo:hi]
    if len(cand) == 0: return []
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey = rep_key[lo:hi]

    agg = {}
    for mz, it in specs:
        qm, qp = clean_spectrum(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0):
                agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]

# MetFrag-lite: In-silico 1-2 bond fragmentation
def mol_graph(smi):
    if not HAVE_RDKIT: return None
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n

def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps = []
    for s in range(n):
        if seen[s]: continue
        stack = [s]; seen[s] = True; cur = [s]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps

def fragment_masses(smi, max_breaks=2, max_bonds=None):
    if max_bonds is None: max_bonds = CFG.MAX_BONDS
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i + 1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))

def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh * H_ATOM + (PROTON_MASS if mode > 0 else -PROTON_MASS))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx + off, 0, len(ion) - 1)
        ok |= np.abs(ion[k] - peak_mz) <= tol
    return float(w[ok].sum() / tot)

def _frag_masses_wrapper(smi):
    try: return fragment_masses(smi)
    except Exception: return np.zeros(0)

def frag_scores(cand_smiles, specs, mode, workers=4):
    if not HAVE_RDKIT:
        return np.zeros(len(cand_smiles), np.float32)
    with MPool(workers) as mp:
        frags = mp.map(_frag_masses_wrapper, cand_smiles, chunksize=8)
    peaks = []
    for mz, it in specs:
        m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
    out = np.zeros(len(cand_smiles), np.float32)
    for j, f in enumerate(frags):
        out[j] = max((explain_score(f, a, b, mode=mode, tol=CFG.MZ_TOL) for a, b in peaks), default=0.0)
    return out

## Simulator helpers (embedded from `build/simsim_common.py`)

In [ ]:
"""Shared helpers for the specs_simple_ms_simulator Phase-1 notebooks.

Inlined verbatim into the generated notebooks (at the SIMSIM_COMMON placeholder) by build/mk_simple_sim_grid.py,
and imported directly by build/test_simple_sim_grid.py. Pure numpy / pandas / pyarrow: no RDKit, no
torch, no ms-pred. `prep` / `esim` call phase1's numba kernels (`_clean`, `entropy_sim`), which the
notebooks embed in an earlier cell; the tests inject them.

Lifted from build/simple_sim_gate.py (sh, sha1, formula_mass, the POOL build, to_model_ce, pb_sf, the
worker call) and from phase1 (ADDUCTS physics, the stage-A/B window logic). The gate notebook itself is
left untouched.
"""
import hashlib
import json
import os
import pickle
import re
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]

# phase1 CFG values (spectrum cleaning and the candidate window) -- unchanged
FLOOR, TOPK, POWER, ENTW, TOL = 0.002, 64, 1.0, True, 0.01
PPM_WIN, PPM_CENTER, PPM_FALLBACK = 8.5, 1.45, 30.0
PPM_EXPAND = (8.5, 20.0, 45.0, 80.0)
TOPN = 25

# --- adduct physics: phase1 cell 3, restricted to the ten test adducts (all n = 1, z = 1) -----------
_M = dict(H=1.00782503207, N=14.0030740048, O=15.9949146196, C=12.0, Na=22.9897692809,
          K=38.96370668, Cl=34.96885268)
_E = 0.00054857990
_PROTON = _M["H"] - _E
_H2O = 2 * _M["H"] + _M["O"]
_NH4 = _M["N"] + 4 * _M["H"]
_FORMATE = _M["C"] + 2 * _M["H"] + 2 * _M["O"]
ADDUCT_DELTA = {
    "[M+H]+": _PROTON, "[M+NH4]+": _NH4 - _E, "[M+Na]+": _M["Na"] - _E, "[M+K]+": _M["K"] - _E,
    "[M-H2O+H]+": _PROTON - _H2O, "[M-2H2O+H]+": _PROTON - 2 * _H2O,
    "[M-H]-": -_PROTON, "[M-H2O-H]-": -_PROTON - _H2O, "[M+CH2O2-H]-": _FORMATE - _PROTON,
    "[M+Cl]-": _M["Cl"] + _E,
}


def neutral_mass1(mz, adduct):
    """ONE precursor m/z -> neutral mass (phase1 neutral_mass, test adducts only; NaN elsewhere).

    Not called `neutral_mass`: the submission notebook also embeds phase1's vectorised neutral_mass.
    """
    d = ADDUCT_DELTA.get(adduct)
    return float(mz) - d if d is not None else np.nan


def precursor_of(mass, adduct):
    """neutral exact mass -> theoretical precursor m/z (the inverse of neutral_mass1)."""
    return np.asarray(mass, dtype=np.float64) + ADDUCT_DELTA[adduct]


# --- label formula -> exact neutral mass (as phase1 / the gate; no RDKit) ---------------------------
_MONO = {"C": 12.0, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221, "S": 31.97207069,
         "P": 30.97376151, "F": 18.99840320, "Cl": 34.96885271, "Br": 78.9183376, "I": 126.904468,
         "Si": 27.9769265327, "Se": 79.9165196, "B": 11.0093055, "Na": 22.98976928, "K": 38.96370649,
         "Fe": 55.9349375, "D": 2.0141017778, "As": 74.9215965, "Li": 7.0160034, "Sn": 119.9021947}
_TOK = re.compile(r"([A-Z][a-z]?)(\d*)")


def formula_mass(f):
    if not isinstance(f, str):
        return np.nan
    s, pos = 0.0, 0
    for m in _TOK.finditer(f):
        if m.start() != pos:
            return np.nan
        pos = m.end()
        if m.group(1) not in _MONO:
            return np.nan
        s += _MONO[m.group(1)] * (int(m.group(2)) if m.group(2) else 1)
    return s if pos == len(f) else np.nan


# --- shell / files ------------------------------------------------------------------------------------
def sh(cmd, timeout=None, env=None):
    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout, env=env)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"command failed ({r.returncode})")
    return r.stdout


def sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# --- ICEBERG worker call (the worker runs in its own venv; files only) -------------------------------
def to_model_ce(ce_ev, prec, mode):
    return ce_ev if mode == "ev" else ce_ev * 500.0 / prec     # ms-pred's own ev_to_nce


def write_request(req, path):
    req[["key", "smiles", "adduct", "precursor_mz", "ce", "instrument"]].to_csv(path, sep="\t", index=False)


def worker_cmd(py, worker, gen, inten, inp, out, device="cuda", batch_size=128, sparse_k=100, prec_margin=2.0):
    return [str(py), str(worker), "--gen", str(gen), "--inten", str(inten), "--inp", str(inp), "--out", str(out),
            "--device", device, "--batch-size", str(batch_size), "--sparse-k", str(sparse_k),
            "--prec-margin", str(prec_margin)]


def read_response(out, n):
    """-> spectra [(mz, it)], ok, err, timing"""
    z = np.load(out)
    # read each array ONCE: NpzFile re-reads (and re-allocates) an array on every z[key] access, and a
    # slice keeps its whole copy alive -- per-row access held n full copies (the Phase-1 Colab OOM)
    off, mz, it = z["off"], z["mz"], z["it"]
    assert len(off) == n + 1, f"{out}: {len(off) - 1} rows, expected {n}"
    spectra = [(mz[off[i]:off[i + 1]], it[off[i]:off[i + 1]]) for i in range(n)]
    return spectra, z["ok"], z["err"], json.loads(str(z["timing"]))


def run_worker(req, work, tag, py, worker, gen, inten, env, **kw):
    """-> spectra, ok, err, timing, wall_s"""
    inp, out = Path(work) / f"req_{tag}.tsv", Path(work) / f"resp_{tag}.npz"
    write_request(req, inp)
    t = time.time()
    r = subprocess.run(worker_cmd(py, worker, gen, inten, inp, out, **kw), text=True, capture_output=True,
                       env=env, timeout=12 * 3600)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-4000:])
        raise RuntimeError(f"worker failed on {tag}")
    spectra, ok, err, timing = read_response(out, len(req))
    return spectra, ok, err, timing, time.time() - t


def err_counts(err):
    return pd.Series([str(e).split(":")[0] for e in err if e]).value_counts().to_dict()


# --- spectra -----------------------------------------------------------------------------------------
def prep(mz, it, prec):
    """phase1 cleaning after the precursor + 2 Da cut. Needs the phase1 kernels in globals."""
    mz = np.asarray(mz, np.float32)
    it = np.asarray(it, np.float32)
    keep = mz <= prec + 2.0
    return _clean(mz[keep], it[keep], FLOOR, TOPK, POWER, ENTW)  # noqa: F821 (phase1 kernel cell)


def esim(a, b):
    if len(a[0]) == 0 or len(b[0]) == 0:
        return 0.0
    return float(entropy_sim(a[0], a[1], b[0], b[1], TOL))  # noqa: F821 (phase1 kernel cell)


def sum_merge(specs, decimals=4):
    """Merge per-CE predicted spectra the way a merged acquisition sums them.

    Each input is (mz, it) with base peak 1. Peaks with the same m/z (rounded to `decimals`; the same
    fragment formula gives the same m/z at every CE) are summed, then the result is renormalised to
    base 1 and sorted by m/z. Empty inputs are ignored.
    """
    parts = [(np.asarray(m, np.float64), np.asarray(i, np.float64)) for m, i in specs if len(m)]
    if not parts:
        return np.empty(0, np.float32), np.empty(0, np.float32)
    mz = np.concatenate([m for m, _ in parts])
    it = np.concatenate([i / i.max() for _, i in parts])
    key, inv = np.unique(np.round(mz, decimals), return_inverse=True)
    s = np.bincount(inv, weights=it)
    m = np.bincount(inv, weights=mz) / np.bincount(inv)
    o = np.argsort(m)
    return m[o].astype(np.float32), (s[o] / s.max()).astype(np.float32)


def rank_of_truth(sims):
    """sims[0] = truth, sims[1:] = decoys. Ties count half; a tied truth is not top-1."""
    sims = np.asarray(sims, dtype=np.float64)
    t, dec = sims[0], sims[1:]
    gt, eq = int((dec > t).sum()), int((dec == t).sum())
    return 1 + gt + eq / 2, (gt == 0 and eq == 0)


def pb_sf(ps, k):
    """P(X >= k), X = sum of independent Bernoulli(p_i) -- exact Poisson-binomial by DP."""
    d = np.zeros(len(ps) + 1)
    d[0] = 1.0
    for p in ps:
        d[1:] = d[1:] * (1 - p) + d[:-1] * p
        d[0] *= (1 - p)
    return float(d[k:].sum())


# --- collision-energy grid ----------------------------------------------------------------------------
def ce_tuple(v):
    """collision_energy_ev cell (list / array / None) -> tuple of floats rounded to 0.1 eV."""
    if v is None:
        return ()
    try:
        return tuple(round(float(x), 1) for x in v)
    except TypeError:
        return ()


def snap_ce(ce, grid, tol):
    """Nearest grid CE within tol (eV), else None. Ties go to the lower CE."""
    if not grid or not np.isfinite(ce):
        return None
    g = min(sorted(grid), key=lambda x: abs(x - ce))
    return float(g) if abs(g - ce) <= tol + 1e-9 else None


def spectrum_conditions(adduct, ces, grid, tol, covered):
    """One spectrum -> set of (adduct, ce, on_grid) conditions that need a simulated spectrum.

    Uncovered adducts need none (MetFrag-lite). A merged multi-CE spectrum needs each constituent CE,
    which is why merges add no grid conditions. CE <= 0 constituents (a few libraries list 0) are skipped.
    """
    if adduct not in covered:
        return set()
    out = set()
    for ce in ces:
        if not np.isfinite(ce) or ce <= 0:
            continue
        g = snap_ce(ce, grid, tol)
        out.add((adduct, g if g is not None else round(float(ce), 1), g is not None))
    return out


def choose_tolerance(median_by_delta, min_sim=0.90):
    """Largest delta (eV) whose median sim(pred@c, pred@c+delta) is >= min_sim; 0.0 if none."""
    ok = [d for d, m in median_by_delta.items() if np.isfinite(m) and m >= min_sim]
    return float(max(ok)) if ok else 0.0


def base_ce_set(ce_tuples, min_cover=0.95):
    """Distinct single CEs covering >= min_cover of single-CE spectra, plus every merged constituent."""
    singles = pd.Series([t[0] for t in ce_tuples if len(t) == 1 and t[0] > 0])
    keep = set()
    if len(singles):
        vc = singles.value_counts()
        cum = vc.cumsum() / vc.sum()
        keep = set(vc.index[: int((cum < min_cover).sum()) + 1])
    for t in ce_tuples:
        if len(t) > 1:
            keep |= {c for c in t if c > 0}
    return sorted(float(c) for c in keep)


# --- candidate pool and windows ------------------------------------------------------------------------
def build_pool(coco_dir, train_path):
    """COCONUT + train structures, deduplicated on ik14, sorted by exact neutral mass (as the gate).

    Columns: k14, smiles, mass, has_real (the structure has >= 1 training spectrum).
    """
    coco_dir = Path(coco_dir)
    cm = pickle.load(open(coco_dir / "coco_meta.pkl", "rb"))
    cmass = np.load(coco_dir / "coco_mass.npy")
    tr = pq.read_table(train_path, columns=["inchikey14", "normalized_smiles", "molecular_formula"]).to_pandas()
    tr = tr.dropna(subset=["inchikey14", "normalized_smiles"]).drop_duplicates("inchikey14")
    fm = {f: formula_mass(f) for f in set(tr.molecular_formula.astype(object).tolist())}
    tr_mass = np.array([fm.get(f, np.nan) for f in tr.molecular_formula.astype(object)], dtype=np.float64)
    pool = pd.DataFrame({
        "k14": np.concatenate([[str(k)[:14] for k in cm["keys"]], tr.inchikey14.astype(str).to_numpy()]),
        "smiles": np.concatenate([np.asarray(cm["smiles"], dtype=object),
                                  tr.normalized_smiles.astype(object).to_numpy()]),
        "mass": np.concatenate([cmass.astype(np.float64), tr_mass]),
    })
    n_raw = len(pool)
    pool = (pool[np.isfinite(pool.mass) & pool.smiles.notna()].drop_duplicates("k14")
            .sort_values("mass", kind="stable").reset_index(drop=True))
    pool["has_real"] = pool.k14.isin(set(tr.inchikey14.astype(str)))
    print(f"pool {n_raw:,} rows -> {len(pool):,} unique ik14 with a finite mass "
          f"(COCONUT {len(cmass):,}, train {len(tr):,})")
    return pool


def window(pm, target, ppm):
    """(lo, hi) index range of the mass-sorted pool within +-ppm of target (phase1 CandidatePool.window)."""
    lo = int(np.searchsorted(pm, target * (1 - ppm / 1e6), "left"))
    hi = int(np.searchsorted(pm, target * (1 + ppm / 1e6), "right"))
    return lo, hi


def scored_range(pm, target, win=PPM_WIN, fallback=PPM_FALLBACK, expand=PPM_EXPAND, topn=TOPN):
    """Pool range whose candidates phase1's stages A + B actually score (stage C only mass-sorts).

    Stage A: the +-win window, or +-fallback if that is empty. Stage B: while fewer than topn candidates
    (pool keys are unique), every wider tier's new candidates are scored. The windows are nested, so the
    scored set is one contiguous range.
    """
    lo, hi = window(pm, target, win)
    if hi == lo:
        lo, hi = window(pm, target, fallback)
    if hi - lo < topn:
        for ppm in expand[1:]:
            a, b = window(pm, target, ppm)
            lo, hi = min(lo, a), max(hi, b)
            if hi - lo >= topn:
                break
    return lo, hi


def query_target(precs, adducts):
    """Molecule -> centred neutral target mass (phase1: median over spectra, divided by 1 + PPM_CENTER)."""
    nms = np.array([neutral_mass1(p, a) for p, a in zip(precs, adducts)], dtype=np.float64)
    nms = nms[np.isfinite(nms)]
    return float(np.median(nms)) / (1.0 + PPM_CENTER / 1e6) if len(nms) else np.nan


# --- projection and decision ---------------------------------------------------------------------------
def on_the_fly(conds, precomputed_adducts):
    """Conditions left for Kaggle: off-grid ones, and on-grid ones whose adduct was not precomputed."""
    return [c for c in conds if not (c[2] and c[0] in precomputed_adducts)]


def bootstrap_total(per_mol, n_mol=400, n_boot=2000, seed=0):
    """Total over n_mol molecules resampled from per_mol -> dict(mean, p50, p95)."""
    x = np.asarray(per_mol, dtype=np.float64)
    if len(x) == 0:
        return dict(mean=0.0, p50=0.0, p95=0.0)
    rng = np.random.default_rng(seed)
    tot = x[rng.integers(0, len(x), size=(n_boot, n_mol))].sum(1)
    return dict(mean=float(x.mean() * n_mol), p50=float(np.percentile(tot, 50)), p95=float(np.percentile(tot, 95)))


def kaggle_hours(load, rate, n_mol, t_base_s, fixed_s, simlib_gb, load_s_per_gb, worker_calls, startup_s):
    """Projected Kaggle wall time (h): fixed setup + per-molecule base cost + simlib load + on-the-fly sims."""
    sim_s = (load / rate if load > 0 else 0.0) + (worker_calls * startup_s if load > 0 else 0.0)
    return (fixed_s + n_mol * t_base_s + simlib_gb * load_s_per_gb + sim_s) / 3600.0


def decide(options, budget_h):
    """Cheapest option (colab_h, then name) whose kaggle_h <= budget_h; None if none fits (-> STOP)."""
    fit = [o for o in options if o["kaggle_h"] <= budget_h]
    return min(fit, key=lambda o: (o["colab_h"], o["name"])) if fit else None


# --- submission (Phase 4, option A: every covered condition simulated on the fly) ---------------------
MERGE_RULES = ("max", "sum", "mid")


def spectrum_ces(ces, rule, default):
    """One query spectrum's CE list -> (CEs to simulate, defaulted).

    CE <= 0 constituents are dropped (a few lists start at 0). `max` / `sum` simulate every constituent;
    `mid` simulates the mean of the list once. No usable CE -> [default], defaulted = True.
    """
    assert rule in MERGE_RULES, rule
    v = [round(float(c), 1) for c in ces if np.isfinite(c) and c > 0]
    if not v:
        return [float(default)], True
    if rule == "mid":
        return [round(float(np.mean(v)), 1)], False
    return sorted(set(v)), False


def sim_priority(lv, fr, ppm):
    """Per-candidate simulation priority (0 = first): lv desc, fr desc, |ppm| asc -- the watchdog's cap order."""
    o = np.lexsort((np.asarray(ppm, np.float64), -np.asarray(fr, np.float64), -np.asarray(lv, np.float64)))
    r = np.empty(len(o), np.int64)
    r[o] = np.arange(len(o))
    return r


REQ_COLS = ["key", "smiles", "adduct", "precursor_mz", "ce_ev", "prio", "ce", "instrument"]


def build_requests(parts, instrument="QTOF", ce_mode="ev"):
    """Molecule parts -> one worker request per distinct (key, adduct, ce_ev), sorted by priority.

    Each part is a dict with arrays key / smiles / mass / prio (one entry per candidate to simulate) and
    conds, a list of (adduct, ce_ev). The precursor is the candidate's theoretical m/z (exact mass +
    adduct delta, as the G2b gate), so a request depends on the candidate alone and is shared by every
    spectrum and molecule that needs it; a shared request keeps its best (lowest) priority.
    """
    frames = []
    for p in parts:
        if not len(p["key"]) or not p["conds"]:
            continue
        mass = np.asarray(p["mass"], np.float64)
        for adduct, ce in p["conds"]:
            frames.append(pd.DataFrame({"key": np.asarray(p["key"], dtype=object),
                                        "smiles": np.asarray(p["smiles"], dtype=object), "adduct": adduct,
                                        "precursor_mz": precursor_of(mass, adduct), "ce_ev": float(ce),
                                        "prio": np.asarray(p["prio"], np.int64)}))
    if not frames:
        return pd.DataFrame({c: [] for c in REQ_COLS})
    req = (pd.concat(frames, ignore_index=True)
           .sort_values(["prio", "key", "adduct", "ce_ev"], kind="stable")
           .drop_duplicates(["key", "adduct", "ce_ev"], keep="first").reset_index(drop=True))
    req["ce"] = to_model_ce(req.ce_ev.to_numpy(), req.precursor_mz.to_numpy(), ce_mode)
    req["instrument"] = instrument
    return req[REQ_COLS]


def rows_that_fit(remaining_s, rate, startup_s):
    """How many worker rows one more call can take before the deadline (0 if none)."""
    return max(0, int((remaining_s - startup_s) * rate))


def spectrum_sv(q, sims, rule):
    """Simulated-library score of ONE prepped query spectrum against one candidate.

    sims: the candidate's raw predictions (mz, it, prec) at each of the spectrum's CEs, None where it was
    not simulated (failed, capped). max / mid: best esim over the CEs; sum: esim against the renormalised
    sum of the available CEs (a merged acquisition). Nothing available -> 0.
    """
    have = [s for s in sims if s is not None and len(s[0])]
    if not have or len(q[0]) == 0:
        return 0.0
    if rule == "sum":
        mz, it = sum_merge([(m, i) for m, i, _ in have])
        return esim(q, prep(mz, it, have[0][2]))
    return max(esim(q, prep(m, i, p)) for m, i, p in have)


def rank_two_tier(lv, sv, fr, ppm, tau):
    """Order (indices) of one candidate set.

    Tier 1 = real-library match lv >= tau, by lv desc (then sv, fr, |ppm|). Tier 2 = the rest, by sv desc,
    fr desc, |ppm| asc. A missing sv / fr (NaN) counts 0; a missing ppm sorts last.
    """
    lv, sv, fr = (np.nan_to_num(np.asarray(x, np.float64), nan=0.0) for x in (lv, sv, fr))
    ppm = np.nan_to_num(np.asarray(ppm, np.float64), nan=np.inf)
    t1, t2 = np.flatnonzero(lv >= tau), np.flatnonzero(lv < tau)
    o1 = t1[np.lexsort((ppm[t1], -fr[t1], -sv[t1], -lv[t1]))]
    o2 = t2[np.lexsort((ppm[t2], -fr[t2], -sv[t2]))]
    return np.concatenate([o1, o2]).astype(np.int64)

## ICEBERG: offline install into its own venv, worker, SHA1, smoke + 120 eV probe

In [ ]:
t = time.time()
UP = os.path.dirname(find_dir("wheelhouse"))
WH = os.path.join(UP, "wheelhouse")
MAN = json.load(open(os.path.join(UP, "manifest.json")))
# manifest covered_adducts is the stale G2 list; CFG.COVERED (G2b) is authoritative
assert MAN["ce_mode"] == CFG.CE_MODE and MAN["instrument"] == CFG.INSTRUMENT, MAN
WORK = Path("/kaggle/working/sim")
WORK.mkdir(parents=True, exist_ok=True)

VENV = "/kaggle/working/iceberg_venv"
PY = f"{VENV}/bin/python"
site_pkg = os.path.join(VENV, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
ENV["PYTHONPATH"] = site_pkg
ENV["DGLBACKEND"] = "pytorch"
ENV["CUDA_VISIBLE_DEVICES"] = "0"          # t4.json: a second T4 adds nothing (CPU-bound)
if not os.path.exists(PY):
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", VENV], check=True)
os.makedirs(site_pkg, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", site_pkg,
                    *sorted(glob.glob(f"{WH}/*.whl"))], capture_output=True, text=True, env=ENV)
assert r.returncode == 0, r.stderr[-3000:]

WORKER_SRC = '"""ICEBERG forward-simulation worker for specs_simple_ms_simulator.\n\nRuns INSIDE the isolated ms-pred environment (its own torch 2.6 / DGL / rdkit 2025.03), never in\nthe notebook\'s main interpreter -- that keeps RDKit 2026.03.3 (the metric\'s version) untouched.\nThe notebook talks to it through files only:\n\n    request tsv      : key, smiles, adduct (CASMI spelling), precursor_mz, ce, instrument\n                       (TSV, not parquet: the ms-pred environment ships no pyarrow)\n    response npz     : off (n+1,) int64, mz / it float32 flat, ok bool, err str, timing json,\n                       above_n int32 / above_share float32 / above_off float32 (per row, see below)\n\nPeaks are the top `--sparse-k` by intensity, sorted by m/z, base peak = 1.0. m/z values are the\ncharged fragment masses (ICEBERG applies the adduct shift itself).\n\nPeaks above precursor_mz + `--prec-margin` (2 Da) are dropped BEFORE the top-k, the same cleaning the\norganisers applied to the test spectra. They exist because the MassSpecGym inten checkpoints have\n`include_unshifted_mz=True`: every fragment is also emitted without the adduct shift, and for\nwater-loss adducts the unshifted root lands 17-35 Da above the precursor. What was dropped is\nrecorded per row: above_n = peak count, above_share = share of raw intensity, above_off = the\nlargest dropped m/z minus the precursor (NaN when nothing was dropped). A row left with no peaks\ngets err "empty_after_prec_cut".\n\nReused unchanged by Phase 0 (gate), Phase 2 (precompute) and Phase 4 (Kaggle on-the-fly).\n\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --inp req.tsv --out resp.npz\n    python iceberg_worker.py --gen G.ckpt --inten I.ckpt --hparams-only --out hp.json\n"""\nimport argparse\nimport json\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom rdkit import Chem, RDLogger\n\nimport ms_pred.common as common\nfrom ms_pred.iceberg import gen_model, inten_model, joint_model\n\nRDLogger.DisableLog("rdApp.*")\n\n# CASMI adduct spelling -> ms-pred spelling (only where they differ)\nADDUCT_MAP = {"[M-2H2O+H]+": "[M+H-2H2O]+", "[M+CH2O2-H]-": "[M+CHO2]-"}\n\n\ndef get_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--gen", required=True)\n    p.add_argument("--inten", required=True)\n    p.add_argument("--inp")\n    p.add_argument("--out", required=True)\n    p.add_argument("--device", default="cuda")\n    p.add_argument("--batch-size", type=int, default=64)\n    p.add_argument("--max-nodes", type=int, default=100)\n    p.add_argument("--sparse-k", type=int, default=100)\n    p.add_argument("--threshold", type=float, default=0.0)\n    p.add_argument("--prec-margin", type=float, default=2.0)\n    p.add_argument("--hparams-only", action="store_true")\n    return p.parse_args()\n\n\ndef _jsonable(d):\n    out = {}\n    for k, v in dict(d).items():\n        try:\n            json.dumps(v)\n            out[k] = v\n        except TypeError:\n            out[k] = str(v)\n    return out\n\n\n# PyTorch 2.6 defaults weights_only=True, which breaks legacy Lightning checkpoints containing PosixPath/hparams.\ntry:\n    from pathlib import PosixPath, WindowsPath\n    torch.serialization.add_safe_globals([PosixPath, WindowsPath])\nexcept Exception:\n    pass\n\n_orig_torch_load = torch.load\ndef _safe_torch_load(*load_args, **load_kwargs):\n    if "weights_only" not in load_kwargs:\n        load_kwargs["weights_only"] = False\n    return _orig_torch_load(*load_args, **load_kwargs)\ntorch.load = _safe_torch_load\n\n\ndef load(args):\n    try:\n        gen = gen_model.FragGNN.load_from_checkpoint(args.gen, map_location="cpu", weights_only=False)\n    except TypeError:\n        gen = gen_model.FragGNN.load_from_checkpoint(args.gen, map_location="cpu")\n\n    try:\n        inten = inten_model.IntenGNN.load_from_checkpoint(args.inten, map_location="cpu", weights_only=False)\n    except TypeError:\n        inten = inten_model.IntenGNN.load_from_checkpoint(args.inten, map_location="cpu")\n\n    return gen, inten\n\n\ndef to_peaks(s, k, prec, margin):\n    """(n, 2) array [mz, inten] -> (mz, it, above_n, above_share, above_off).\n\n    Drops peaks above prec + margin, then keeps the top-k by intensity, sorted by m/z, base = 1.\n    """\n    s = np.asarray(s, dtype=np.float64).reshape(-1, 2)\n    s = s[np.isfinite(s).all(axis=1) & (s[:, 1] > 0)]\n    above = s[:, 0] > prec + margin\n    above_n = int(above.sum())\n    above_share = float(s[above, 1].sum() / s[:, 1].sum()) if len(s) else 0.0\n    above_off = float(s[above, 0].max() - prec) if above_n else np.nan\n    s = s[~above]\n    if len(s) == 0:\n        return np.empty(0, np.float32), np.empty(0, np.float32), above_n, above_share, above_off\n    s = s[np.argsort(s[:, 1])[::-1][:k]]\n    s = s[np.argsort(s[:, 0])]\n    return (s[:, 0].astype(np.float32), (s[:, 1] / s[:, 1].max()).astype(np.float32),\n            above_n, above_share, above_off)\n\n\ndef predict_batch(model, rows, args):\n    return model.predict_mol(\n        [r["smiles"] for r in rows],\n        collision_eng=[float(r["ce"]) for r in rows],\n        precursor_mz=[float(r["precursor_mz"]) for r in rows],\n        adduct=[r["adduct"] for r in rows],\n        instrument=[r["instrument"] for r in rows],\n        threshold=args.threshold,\n        device=args.device,\n        max_nodes=args.max_nodes,\n        adduct_shift=False,\n        canonical_root_smi=True,\n        binned_out=False,\n    )["spec"]\n\n\ndef main():\n    args = get_args()\n    t0 = time.time()\n    gen, inten = load(args)\n\n    if args.hparams_only:\n        with open(args.out, "w") as f:\n            json.dump({"gen": _jsonable(gen.hparams), "inten": _jsonable(inten.hparams),\n                       "ion2onehot_pos": common.ion2onehot_pos,\n                       "instrument2onehot_pos": common.instrument2onehot_pos}, f, indent=1)\n        return\n\n    model = joint_model.JointModel(gen_model_obj=gen, inten_model_obj=inten).to(args.device)\n    model.eval()\n    model.freeze()\n    t_load = time.time() - t0\n\n    df = pd.read_csv(args.inp, sep="\t")\n    n = len(df)\n    peaks = [None] * n\n    err = [""] * n\n    above_n = np.zeros(n, np.int32)\n    above_share = np.zeros(n, np.float32)\n    above_off = np.full(n, np.nan, np.float32)\n\n    # canonicalise exactly as predict_smis.py does; reject what ms-pred cannot take\n    rows = []\n    for i, r in enumerate(df.to_dict("records")):\n        m = Chem.MolFromSmiles(r["smiles"]) if isinstance(r["smiles"], str) else None\n        ad = ADDUCT_MAP.get(r["adduct"], r["adduct"])\n        if m is None:\n            err[i] = "rdkit_parse"\n        elif ad not in common.ion2onehot_pos:\n            err[i] = f"adduct_unsupported:{r[\'adduct\']}"\n        elif r["instrument"] not in common.instrument2onehot_pos:\n            err[i] = f"instrument_unsupported:{r[\'instrument\']}"\n        elif not np.isfinite(r["ce"]):\n            err[i] = "ce_nan"\n        else:\n            rows.append(dict(r, i=i, smiles=Chem.MolToSmiles(m), adduct=ad))\n\n    t1 = time.time()\n    with torch.no_grad():\n        for b in range(0, len(rows), args.batch_size):\n            batch = rows[b:b + args.batch_size]\n            try:\n                specs = predict_batch(model, batch, args)\n            except Exception:\n                # isolate the failing molecule(s); keep the rest of the batch\n                specs = []\n                for r in batch:\n                    try:\n                        specs.append(predict_batch(model, [r], args)[0])\n                    except Exception as e:  # noqa: BLE001\n                        specs.append(None)\n                        err[r["i"]] = f"predict:{type(e).__name__}:{str(e)[:120]}"\n            for r, s in zip(batch, specs):\n                if s is not None:\n                    i = r["i"]\n                    mz_, it_, above_n[i], above_share[i], above_off[i] = to_peaks(\n                        s.detach().cpu().numpy(), args.sparse_k, float(r["precursor_mz"]), args.prec_margin)\n                    peaks[i] = (mz_, it_)\n    if args.device.startswith("cuda"):\n        torch.cuda.synchronize()\n    t_pred = time.time() - t1\n\n    empty = (np.empty(0, np.float32), np.empty(0, np.float32))\n    ok = np.array([p is not None and len(p[0]) > 0 for p in peaks])\n    for i in range(n):\n        if peaks[i] is not None and len(peaks[i][0]) == 0 and not err[i]:\n            err[i] = "empty_after_prec_cut" if above_n[i] else "empty_spectrum"\n    peaks = [p if p is not None else empty for p in peaks]\n    lens = np.array([len(p[0]) for p in peaks], dtype=np.int64)\n    off = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    mz = np.concatenate([p[0] for p in peaks]) if n else np.empty(0, np.float32)\n    it = np.concatenate([p[1] for p in peaks]) if n else np.empty(0, np.float32)\n    timing = dict(n=n, n_predicted=int(ok.sum()), t_load=t_load, t_pred=t_pred,\n                  mol_per_s=float(len(rows) / t_pred) if t_pred > 0 else None,\n                  device=args.device, batch_size=args.batch_size, prec_margin=args.prec_margin,\n                  gpu=torch.cuda.get_device_name(0) if args.device.startswith("cuda") else "cpu")\n    np.savez(args.out, off=off, mz=mz, it=it, ok=ok, err=np.array(err, dtype="U200"),\n             above_n=above_n, above_share=above_share, above_off=above_off, timing=json.dumps(timing))\n    print(json.dumps(timing))\n\n\nif __name__ == "__main__":\n    main()\n'
WORKER = WORK / "iceberg_worker.py"
WORKER.write_text(WORKER_SRC)
GEN, INTEN = os.path.join(UP, "ckpt", "gen.ckpt"), os.path.join(UP, "ckpt", "inten.ckpt")
assert sha1(GEN) == MAN["gen_sha1"] and sha1(INTEN) == MAN["inten_sha1"], "checkpoint SHA1 mismatch"
DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
if DEVICE == "cpu":
    print("[WARNING] no GPU -- ICEBERG on CPU is ~15x slower; the watchdog will cap the simulation")

def sim_call(req, tag, bs=None):
    return run_worker(req, WORK, tag, PY, WORKER, GEN, INTEN, ENV, device=DEVICE,
                      batch_size=bs or CFG.BATCH, sparse_k=CFG.SPARSE_K)

# smoke (caffeine, [M+H]+ 40 eV) must predict; the off-ladder probe (120 eV) goes into the report
_m = 194.0803756
SMOKE = pd.DataFrame({"key": ["RYYVLZVUVIJVGH", "RYYVLZVUVIJVGH"], "smiles": ["Cn1cnc2c1c(=O)n(C)c(=O)n2C"] * 2,
                      "adduct": "[M+H]+", "precursor_mz": precursor_of(np.array([_m, _m]), "[M+H]+"),
                      "ce_ev": [40.0, 120.0]})
SMOKE["ce"] = to_model_ce(SMOKE.ce_ev.to_numpy(), SMOKE.precursor_mz.to_numpy(), CFG.CE_MODE)
SMOKE["instrument"] = CFG.INSTRUMENT
_sp, _ok, _err, _tm, _wall = sim_call(SMOKE, "smoke")
PROBE_120 = dict(ok=bool(_ok[1]), n_peaks=int(len(_sp[1][0])), err=str(_err[1]))
assert _ok[0], f"ICEBERG smoke failed: {_err[0]}"          # the probe is reported, not asserted
TIMES["iceberg_setup_s"] = round(time.time() - t, 1)
print(f"[SUCCESS] ICEBERG ready on {_tm['gpu']} in {TIMES['iceberg_setup_s']} s (SHA1 OK); "
      f"smoke {len(_sp[0][0])} peaks @40 eV, probe {PROBE_120['n_peaks']} peaks @120 eV")

## Load: candidate pool, reference library, test; run-mode detection

In [ ]:
t = time.time()
pool = build_candidate_pool()
TIMES["pool_s"] = round(time.time() - t, 1)
t = time.time()
L = load_library(TRAIN)
TIMES["library_s"] = round(time.time() - t, 1)

samp = pd.read_csv(SAMPLE)
te = pq.read_table(TEST).to_pandas()
te['nm'] = neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id'))

# Run-mode detection (phase1) - FAIL-SAFE: full run unless the PUBLIC test file is proven
_env_raw  = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
env_rerun = bool(_env_raw)                       # truthiness, never an exact string compare
n_mol     = int(te.molecule_id.nunique())
looks_public = (n_mol == CFG.PUBLIC_N_MOL and len(te) == CFG.PUBLIC_N_SPEC)
FAST_MODE    = bool(CFG.ALLOW_FAST) and looks_public and not env_rerun and not CFG.FORCE_FULL
IS_FULL_RUN  = not FAST_MODE
SIM_TIER1    = CFG.SIM_TIER1 or FAST_MODE        # the fast commit needs tier-1 sims for the plumbing check

print("=" * 78, flush=True)
print(f"  KAGGLE_IS_COMPETITION_RERUN = {_env_raw!r} | test rows = {len(te):,} | molecules = {n_mol:,}", flush=True)
if IS_FULL_RUN:
    print(f"  [MODE: FULL RUN]  two-tier engine on all {n_mol:,} molecules. This output is SUBMITTABLE.", flush=True)
else:
    print(f"  [MODE: FAST COMMIT]  full engine on the first {CFG.FAST_DEV_MOLECULES} molecules only.", flush=True)
    print( "  *** THIS OUTPUT IS NOT A REAL SUBMISSION - public test file detected ***", flush=True)
print("=" * 78, flush=True)
print(f"[CONFIG] USE_SIM={CFG.USE_SIM} TAU={CFG.TAU} MERGE_RULE={CFG.MERGE_RULE} COVERED={list(CFG.COVERED)} "
      f"SIM_TIER1={SIM_TIER1} BUDGET={CFG.HARD_BUDGET_H}h", flush=True)

## Pass 1 (CPU): windows, real-library `lv`, MetFrag-lite `fr`, covered conditions → simulation requests

The scored set is phase1's stage A (±8.5 ppm, or ±30 if empty) plus the PPM_EXPAND tiers until ≥ 25
candidates (`scored_range`). Stage C (mass-nearest top-up) is never scored, so never simulated.

In [ ]:
PLANS = {}
N_CE_DEFAULTED = 0
t = time.time()
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    if not len(nms):
        continue
    target = float(np.median(nms)) / (1.0 + CFG.PPM_CENTER / 1e6)
    deep = IS_FULL_RUN or gi < CFG.FAST_DEV_MOLECULES
    if not deep:
        PLANS[gi] = dict(target=target, deep=False)
        continue
    t_m = time.time()
    specs = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]
    lib_hits = lib_sim(L, specs, target)
    ion_mode = float(np.mean([1.0 if m == 'positive' else -1.0 for m in sub.ionization_mode]))
    lo, hi = scored_range(pool.mass, target)
    cand = np.arange(lo, hi)
    lv = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)
    fr = frag_scores([pool.smiles[c] for c in cand], specs, ion_mode) if len(cand) else np.zeros(0, np.float32)
    ppm = np.abs(pool.mass[cand] / target - 1.0) * 1e6

    qs = []                                        # covered query spectra: (adduct, CEs to simulate, prepped)
    for r in sub.itertuples():
        if r.adduct not in CFG.COVERED:
            continue
        ces, defaulted = spectrum_ces(ce_tuple(r.collision_energy_ev), CFG.MERGE_RULE, CFG.CE_DEFAULT)
        N_CE_DEFAULTED += defaulted
        q = prep(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz))
        if len(q[0]):
            qs.append((r.adduct, ces, q))
    conds = sorted({(a, c) for a, ces, _ in qs for c in ces})

    valid = np.array([k is not None and len(str(k)) >= 14 for k in pool.keys[cand]], bool)
    sim = valid & ((lv < CFG.TAU) | SIM_TIER1) & bool(CFG.USE_SIM)
    PLANS[gi] = dict(target=target, deep=True, lib_hits=lib_hits, ion_mode=ion_mode, specs=specs,
                     fr={int(c): float(f) for c, f in zip(cand, fr)}, qs=qs, conds=conds,
                     n_cand=len(cand), n_sim=int(sim.sum()), n_sim_nt1=int((valid & (lv < CFG.TAU)).sum()),
                     sim_idx=cand[sim], prio=sim_priority(lv[sim], fr[sim], ppm[sim]), t_pass1=time.time() - t_m)
    if gi % 50 == 0:
        print(f"  pass 1: {gi + 1}/{len(mols)} | {time.time() - T0:.0f}s", flush=True)

DEEP = [gi for gi, p in PLANS.items() if p["deep"]]
TIMES["pass1_s"] = round(time.time() - t, 1)
REQ = build_requests(({"key": pool.keys[p["sim_idx"]], "smiles": pool.smiles[p["sim_idx"]],
                       "mass": pool.mass[p["sim_idx"]], "prio": p["prio"], "conds": p["conds"]}
                      for p in (PLANS[g] for g in DEEP)), CFG.INSTRUMENT, CFG.CE_MODE)
n_cond = pd.Series([len(PLANS[g]["conds"]) for g in DEEP])
print(f"[PASS 1] {len(DEEP):,} molecules in {TIMES['pass1_s']:.0f}s "
      f"({TIMES['pass1_s'] / max(len(DEEP), 1):.1f} s/mol); candidates/mol median "
      f"{np.median([PLANS[g]['n_cand'] for g in DEEP]):.0f}; covered conditions/mol mean {n_cond.mean():.2f} "
      f"(none: {(n_cond == 0).sum()}); CE defaulted on {N_CE_DEFAULTED} spectra", flush=True)
print(f"[PASS 1] {len(REQ):,} distinct simulation requests "
      f"(adducts {REQ.adduct.value_counts().to_dict()})", flush=True)

## Simulate: priority-ordered chunks with a deadline watchdog

Requests are sorted by each candidate's rank inside its molecule (lv, fr, |ppm|), so every molecule's
best candidates go first. Before each chunk the watchdog checks the deadline with the measured rate; if
the rest does not fit, it runs what fits and stops (the unsimulated rest ranks by `fr` / ppm).

In [ ]:
t = time.time()
reserve_s = CFG.PASS2_S_PER_MOL * len(DEEP) + 300.0
DEADLINE = T0 + CFG.HARD_BUDGET_H * 3600.0 - reserve_s
rate, startup = CFG.T4_RATE, CFG.STARTUP_S
n_chunks = math.ceil(len(REQ) / CFG.CHUNK) if len(REQ) else 0
proj_h = (time.time() - T0 + len(REQ) / rate + n_chunks * startup + reserve_s) / 3600.0
print(f"[WATCHDOG] projected total {proj_h:.2f} h for {len(REQ):,} requests at {rate} mol/s "
      f"(budget {CFG.HARD_BUDGET_H} h){' -> WILL CAP' if proj_h > CFG.HARD_BUDGET_H else ''}", flush=True)

SIM = {}                      # (key, adduct, ce_ev) -> raw prediction (mz, it, precursor)
ERRS, FAILED_ROWS, CAPPED_ROWS, done, ci = {}, 0, 0, 0, 0
while done < len(REQ):
    fit = rows_that_fit(DEADLINE - time.time(), rate, startup)
    n = min(CFG.CHUNK, len(REQ) - done, fit)
    if fit < min(500, len(REQ) - done):                # not even a useful partial chunk fits
        CAPPED_ROWS = len(REQ) - done
        print(f"[WATCHDOG] deadline: {CAPPED_ROWS:,} lowest-priority requests left unsimulated", flush=True)
        break
    part = REQ.iloc[done:done + n]
    try:
        spectra, ok, err, timing, wall = sim_call(part, f"c{ci}")
    except RuntimeError:
        try:
            spectra, ok, err, timing, wall = sim_call(part, f"c{ci}r", bs=CFG.BATCH_RETRY)
        except RuntimeError:
            if FAST_MODE:
                raise                                  # a bug must be visible in the commit
            FAILED_ROWS += n
            print(f"[WARNING] chunk {ci} ({n:,} rows) failed twice -- those candidates rank by fr/ppm", flush=True)
            done, ci = done + n, ci + 1
            continue
    for row, s, o in zip(part.itertuples(), spectra, ok):
        if o:
            SIM[(row.key, row.adduct, row.ce_ev)] = (s[0], s[1], row.precursor_mz)
    for k, v in err_counts(err).items():
        ERRS[k] = ERRS.get(k, 0) + v
    if timing.get("mol_per_s"):
        rate, startup = float(timing["mol_per_s"]), max(float(wall - timing["t_pred"]), 1.0)
    for f in WORK.glob(f"*_c{ci}*"):
        f.unlink()
    done, ci = done + n, ci + 1
    print(f"  sim chunk {ci}/{n_chunks}: {done:,}/{len(REQ):,} rows | {rate:.1f} mol/s | "
          f"ok {len(SIM):,} | {time.time() - T0:.0f}s", flush=True)

TIMES["sim_s"] = round(time.time() - t, 1)
N_ATTEMPTED = len(REQ) - CAPPED_ROWS
SIM_OK = len(SIM) / (N_ATTEMPTED - FAILED_ROWS) if N_ATTEMPTED > FAILED_ROWS else float("nan")
print(f"[SIM] {len(SIM):,}/{len(REQ):,} predicted in {TIMES['sim_s']:.0f}s; ok share {SIM_OK:.4f}; "
      f"errors {ERRS}; failed rows {FAILED_ROWS:,}; capped rows {CAPPED_ROWS:,}", flush=True)

## Pass 2: `sv`, two-tier ranking, phase1 slot filling (stage A → B → C)

In [ ]:
def take_slots(order, cand, smis, seen):
    """Append ranked candidates, strictly InChIKey14-deduplicated, until TOPN slots are full."""
    for idx in order:
        c_idx = cand[idx]
        raw_k = pool.keys[c_idx]
        if raw_k is None:
            continue
        k14 = str(raw_k)[:14]
        if len(k14) == 14 and k14 not in seen:
            seen.add(k14)
            smis.append(pool.smiles[c_idx])
            if len(smis) == CFG.TOPN:
                break

def features(plan, idx, target):
    """lv, sv, fr, ppm for pool indices idx (fr computed now for any index pass 1 did not score)."""
    lv = np.array([plan["lib_hits"].get(pool.keys[c], 0.0) for c in idx], np.float32)
    miss = [int(c) for c in idx if int(c) not in plan["fr"]]
    if miss:
        plan["fr"].update(zip(miss, map(float, frag_scores([pool.smiles[c] for c in miss],
                                                           plan["specs"], plan["ion_mode"]))))
    fr = np.array([plan["fr"][int(c)] for c in idx], np.float32)
    ppm = np.abs(pool.mass[idx] / target - 1.0) * 1e6
    sv = np.zeros(len(idx), np.float32)
    for j, c in enumerate(idx):
        k = pool.keys[c]
        for adduct, ces, q in plan["qs"]:
            sv[j] = max(sv[j], spectrum_sv(q, [SIM.get((k, adduct, ce)) for ce in ces], CFG.MERGE_RULE))
    return lv, sv, fr, ppm

t = time.time()
rows, diag, STAGE_A = [], [], {}
for gi, (mid, sub) in enumerate(mols):
    smis = []
    plan = PLANS.get(gi)
    if plan is not None and plan["deep"]:
        target = plan["target"]
        # -- Stage A: tight centred window. This ranking is authoritative. ---------------------
        cand = pool.window(target, CFG.PPM_WIN)
        if len(cand) == 0:
            cand = pool.window(target, CFG.PPM_FALLBACK)
        seen_keys = set()
        if len(cand):
            lv, sv, fr, ppm = features(plan, cand, target)
            order = rank_two_tier(lv, sv, fr, ppm, CFG.TAU)
            take_slots(order, cand, smis, seen_keys)
            top = order[0]
            diag.append((mid, target, len(cand), len(plan["qs"]), len(plan["conds"]), float(lv.max()),
                         float(sv.max()), float(fr.max()), bool(lv[top] >= CFG.TAU)))
            if FAST_MODE:
                STAGE_A[gi] = (cand, lv, sv, fr, ppm)
        # -- Stage B: widened window, APPEND ONLY. Cannot reorder stage A. ---------------------
        if CFG.EXPAND_BACKFILL and len(smis) < CFG.TOPN:
            for ppm_tier in CFG.PPM_EXPAND[1:]:
                wide = pool.window(target, ppm_tier)
                new_cand = wide[~np.isin(wide, cand)]
                if len(new_cand) == 0:
                    continue
                take_slots(rank_two_tier(*features(plan, new_cand, target), CFG.TAU), new_cand, smis, seen_keys)
                if len(smis) == CFG.TOPN:
                    break
        # -- Stage C: mass-nearest top-up (never blind 'CCO' padding) --------------------------
        if len(smis) < CFG.TOPN:
            wide = pool.window(target, CFG.BACKFILL_PPM)
            if len(wide):
                for c_idx in wide[np.argsort(np.abs(pool.mass[wide] - target))]:
                    k = pool.keys[c_idx]
                    if k is not None and k not in seen_keys and len(str(k)) >= 14:
                        seen_keys.add(k)
                        smis.append(pool.smiles[c_idx])
                        if len(smis) == CFG.TOPN:
                            break
    elif plan is not None:
        # FAST COMMIT FALLBACK (phase1): instant mass-matched candidates from the pool
        target = plan["target"]
        cand = pool.window(target, 25.0)
        if len(cand):
            cand = cand[np.argsort(np.abs(pool.mass[cand] - target))]
            seen_keys = set()
            for c_idx in cand:
                k = pool.keys[c_idx]
                if k not in seen_keys and k is not None:
                    seen_keys.add(k)
                    smis.append(pool.smiles[c_idx])
                    if len(smis) == CFG.TOPN:
                        break
    while len(smis) < CFG.TOPN:
        smis.append('CCO')
    rows.append((mid, ';'.join(smis[:CFG.TOPN])))
TIMES["pass2_s"] = round(time.time() - t, 1)
print(f"[PASS 2] ranked {len(DEEP):,} molecules in {TIMES['pass2_s']:.0f}s", flush=True)

## Submission (phase1 audit)

In [ ]:
submission = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
submission = samp[['molecule_id']].merge(submission, on='molecule_id', how='left')
submission['smiles'] = submission['smiles'].fillna('CCO')

def pad_candidates(s):
    tokens = [t for t in s.split(';') if t]
    if len(tokens) < CFG.TOPN:
        tokens += ['CCO'] * (CFG.TOPN - len(tokens))
    return ';'.join(tokens[:CFG.TOPN])

submission['smiles'] = submission['smiles'].apply(pad_candidates)
assert len(submission) == len(samp), f"Row count mismatch: expected {len(samp)}, got {len(submission)}"
assert (submission['molecule_id'] == samp['molecule_id']).all(), "molecule_id alignment mismatch"
assert submission.molecule_id.duplicated().sum() == 0, "Duplicated molecule_ids found"
assert submission.smiles.isnull().sum() == 0, "Null values present in smiles column"
counts = submission['smiles'].apply(lambda s: len(s.split(';')))
assert (counts == 25).all(), "Every row must have exactly 25 candidates separated by semicolon"
submission.to_csv('submission.csv', index=False)
print(f"[SUCCESS] submission.csv {submission.shape} written at {time.time() - T0:.0f}s")
submission.head()

## Checks and run report (roadmap Phase 5)

On the public test (train examples) nearly every molecule should rank a tier-1 real match first; a low
tier-1 rate means a bug. The tier-2 plumbing check (fast commit) zeroes the tier-1 winner's `lv` —
as if it had no real spectrum — and asks whether `sv` alone still keeps it in the top 25.

In [ ]:
D = pd.DataFrame(diag, columns=["molecule_id", "target", "n_cand", "n_covered_spec", "n_cond",
                                "lv_max", "sv_max", "fr_max", "top_is_tier1"])
T1_RATE = float(D.top_is_tier1.mean()) if len(D) else float("nan")

plumb = []
for gi in [g for g in DEEP if g in STAGE_A][:20]:
    cand, lv, sv, fr, ppm = STAGE_A[gi]
    top = rank_two_tier(lv, sv, fr, ppm, CFG.TAU)[0]
    if lv[top] < CFG.TAU or not PLANS[gi]["qs"]:
        continue
    lv2 = lv.copy()
    lv2[top] = 0.0
    pos = int(np.flatnonzero(rank_two_tier(lv2, sv, fr, ppm, CFG.TAU) == top)[0]) + 1
    plumb.append(pos)

n_full = len(mols)
p1 = TIMES["pass1_s"] / max(len(DEEP), 1)
req_nt1 = np.mean([PLANS[g]["n_sim_nt1"] * len(PLANS[g]["conds"]) for g in DEEP]) if DEEP else 0.0
fixed_s = TIMES["iceberg_setup_s"] + TIMES["pool_s"] + TIMES["library_s"]
proj_full_h = (fixed_s + n_full * (p1 + CFG.PASS2_S_PER_MOL) + n_full * req_nt1 / rate
               + math.ceil(n_full * req_nt1 / CFG.CHUNK) * startup) / 3600.0

REPORT = dict(
    mode="fast" if FAST_MODE else "full", use_sim=bool(CFG.USE_SIM), n_molecules=n_full, n_deep=len(DEEP), tau=CFG.TAU,
    merge_rule=CFG.MERGE_RULE, covered=list(CFG.COVERED), times=TIMES, total_s=round(time.time() - T0, 1),
    n_requests=len(REQ), n_predicted=len(SIM), sim_ok_share=round(SIM_OK, 4), sim_errors=ERRS,
    failed_rows=FAILED_ROWS, capped_rows=CAPPED_ROWS, rate_mol_s=round(rate, 2), startup_s=round(startup, 1),
    ce_defaulted=N_CE_DEFAULTED, probe_120ev=PROBE_120, tier1_rate=T1_RATE,
    tier2_plumbing=dict(n=len(plumb), in_top25=float(np.mean(np.array(plumb) <= 25)) if plumb else None,
                        median_rank=float(np.median(plumb)) if plumb else None),
    per_mol=D[["n_cand", "n_covered_spec", "n_cond", "lv_max", "sv_max"]].describe().round(3).to_dict() if len(D) else {},
    projected_full_run_h=round(proj_full_h, 2),
)
Path("/kaggle/working/run_report.json").write_text(json.dumps(REPORT, indent=1, default=str))
print("=" * 78)
print(json.dumps({k: REPORT[k] for k in ("mode", "n_deep", "times", "total_s", "n_requests", "sim_ok_share",
                                          "capped_rows", "failed_rows", "rate_mol_s", "probe_120ev",
                                          "tier1_rate", "tier2_plumbing", "projected_full_run_h")},
                 indent=1, default=str))
print("=" * 78)
checks = {
    "sim ok share >= 0.99": (SIM_OK >= CFG.MIN_OK_FAST) if CFG.USE_SIM else None,
    "120 eV probe predicted": PROBE_120["ok"],
    "tier-1 rate >= 0.9 (public test only)": (T1_RATE >= 0.9) if FAST_MODE else None,
    "projected full run <= 6.3 h": proj_full_h <= 6.3,
}
for k, v in checks.items():
    print(f"  {'PASS ' if v else ('n/a  ' if v is None else 'CHECK')} {k}")
if FAST_MODE and CFG.USE_SIM:
    assert SIM_OK >= CFG.MIN_OK_FAST, f"only {SIM_OK:.3f} of the simulation requests were predicted: {ERRS}"